In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 02 · EDA — qué hay realmente en mis datos (Eco-Score)

El EDA **no es hacer gráficos**: es descubrir qué decisiones hay que tomar antes de modelar. Este notebook termina con
una **tabla de decisiones**, cada una con una cifra y un motivo. Los hallazgos y la tabla están en `reports/01_eda.md`.

**Preguntas guía:** (1) unidad de análisis · (2) objetivo · (3) suciedad · (4) qué debería ser estable · (5) sospecha de fuga.

> El archivo original `data/raw/prosperLoanData.csv` solo se lee; nunca se modifica.

## 0. Cargar los datos

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RAW_PATH = Path.cwd().resolve().parent / "data" / "raw" / "prosperLoanData.csv"
df = pd.read_csv(RAW_PATH, sep=";", low_memory=False)  # el archivo usa ';' como separador
print(f"{RAW_PATH.name}: {df.shape[0]:,} filas x {df.shape[1]} columnas")

# No hay nombres, documentos ni teléfonos. Las llaves (MemberKey, ListingKey, LoanKey, GroupKey) son seudónimas:
# se conservan aquí para medir repeticiones y se quitan de las variables en el preprocesamiento.
ID_COL = "MemberKey"     # prestatario: puede tener varios préstamos
LOAN_COL = "ListingKey"  # préstamo
TARGET = "is_default"    # se construye en la sección 2 a partir de LoanStatus

prosperLoanData.csv: 113,937 filas x 81 columnas


## 1. ¿Cuál es la unidad de análisis? ¿Qué representa una fila?

*Pregunta:* ¿una fila es un cliente, una transacción, un cliente en un mes...? ¿Alguna entidad aparece en **varias
filas**? De eso depende cómo se parte el dato.

In [3]:
print("filas:", len(df), "| columnas:", df.shape[1])
print(f"préstamos distintos ({LOAN_COL}): {df[LOAN_COL].nunique():,}  -> filas extra por repetición: {df[LOAN_COL].duplicated().sum():,}")
print(f"prestatarios distintos ({ID_COL}): {df[ID_COL].nunique():,}")

loans_per_member = df.groupby(ID_COL)[LOAN_COL].nunique()
multi = loans_per_member[loans_per_member > 1]
print(f"prestatarios con más de un préstamo: {multi.size:,} | filas de esos prestatarios: {df[ID_COL].isin(multi.index).sum():,}")
print(f"máximo de filas por prestatario: {df[ID_COL].value_counts().max()}")
loans_per_member.value_counts().sort_index().rename("prestatarios con N préstamos")

filas: 113937 | columnas: 81
préstamos distintos (ListingKey): 113,048  -> filas extra por repetición: 889
prestatarios distintos (MemberKey): 90,824


prestatarios con más de un préstamo: 15,744 | filas de esos prestatarios: 38,149
máximo de filas por prestatario: 9


ListingKey
1    75080
2    11235
3     3074
4     1022
5      310
6       74
7       21
8        7
9        1
Name: prestatarios con N préstamos, dtype: int64

**Mi respuesta (una frase):** una fila es **un préstamo personal** de Prosper, con su estado al cierre del dataset
(marzo de 2014). Un mismo prestatario puede tener varios préstamos.

**Unidad por la que partiré los datos:** el prestatario (`MemberKey`). Si sus préstamos quedaran repartidos entre
entrenamiento y prueba, el modelo "reconocería" a la persona y la métrica se inflaría.

## 2. ¿Cómo está el objetivo?

*Pregunta:* ¿qué porcentaje acierta predecir siempre la clase más frecuente? ¿El objetivo cambia dentro de la misma
entidad? ¿Tiene vacíos?

`LoanStatus` es la fuente del objetivo. Primero, cómo viene y qué pasaba con la clasificación de `eco_score_pd.ipynb`.

In [4]:
print(df["LoanStatus"].value_counts().to_string())
print()
print("'Charged-Off' (como lo buscaba eco_score_pd.ipynb):", (df["LoanStatus"] == "Charged-Off").sum())
print("'Chargedoff'  (como viene en los datos):          ", (df["LoanStatus"] == "Chargedoff").sum())

LoanStatus
Current                   56576
Completed                 38074
Chargedoff                11992
Defaulted                  5018
Past Due (1-15 days)        806
Past Due (31-60 days)       363
Past Due (61-90 days)       313
Past Due (91-120 days)      304
Past Due (16-30 days)       265
FinalPaymentInProgress      205
Past Due (>120 days)         16
Cancelled                     5

'Charged-Off' (como lo buscaba eco_score_pd.ipynb): 0
'Chargedoff'  (como viene en los datos):           11992


Construimos el objetivo aprobado `is_default` (1 = `Chargedoff` o `Defaulted`, 0 = `Completed`) sobre la población a
modelar: préstamos **terminados**, una fila por `ListingKey` y con `BorrowerState` conocido.

In [5]:
FINISHED = ["Completed", "Chargedoff", "Defaulted"]
DEFAULT = ["Chargedoff", "Defaulted"]

unfinished = ~df["LoanStatus"].isin(FINISHED)
print(f"préstamos sin resultado final (vacío del objetivo): {unfinished.sum():,}")

loans = (df[~unfinished]
         .drop_duplicates(subset=LOAN_COL, keep="first")
         .dropna(subset=["BorrowerState"])
         .copy())
loans[TARGET] = loans["LoanStatus"].isin(DEFAULT).astype(int)
print(f"población a modelar: {len(loans):,} préstamos")
print(loans[TARGET].value_counts().rename({0: "pagó", 1: "impago"}).to_string())
print(f"\nlínea base 'siempre pagó': exactitud = {1 - loans[TARGET].mean():.4f} | recall de impago = 0")
print(f"AUC-PR de un modelo sin información = {loans[TARGET].mean():.4f}")

préstamos sin resultado final (vacío del objetivo): 58,853
población a modelar: 49,552 préstamos
is_default
pagó      34172
impago    15380

línea base 'siempre pagó': exactitud = 0.6896 | recall de impago = 0
AUC-PR de un modelo sin información = 0.3104


In [6]:
# ¿El objetivo cambia dentro del mismo prestatario?
by_member = loans.groupby(ID_COL)[TARGET].agg(["size", "nunique", "mean"])
repeated = by_member[by_member["size"] > 1]
print(f"prestatarios con >1 préstamo terminado: {len(repeated):,} ({repeated['size'].sum():,} préstamos)")
print(f"de ellos, con resultados distintos entre sus préstamos: {(repeated['nunique'] > 1).mean():.1%}")
print(f"tasa de impago en prestatarios con 1 préstamo:  {by_member.loc[by_member['size'] == 1, 'mean'].mean():.1%}")
print(f"tasa de impago en préstamos de prestatarios con >1: {loans[loans[ID_COL].isin(repeated.index)][TARGET].mean():.1%}")

prestatarios con >1 préstamo terminado: 7,028 (16,128 préstamos)
de ellos, con resultados distintos entre sus préstamos: 15.2%


tasa de impago en prestatarios con 1 préstamo:  37.3%
tasa de impago en préstamos de prestatarios con >1: 18.1%


In [7]:
# Tasa de impago por año de originación (orden temporal)
loans["origination_year"] = loans["LoanOriginationQuarter"].str[-4:].astype(int)
pd.DataFrame({"préstamos": loans.groupby("origination_year").size(),
              "impago_%": loans.groupby("origination_year")[TARGET].mean().mul(100).round(1)})

,préstamos,impago_%
origination_year,,
2006,2904,53.1
2007,8997,40.9
2008,11517,33.0
2009,2047,15.2
2010,5625,16.7
2011,7631,28.1
2012,8140,32.2
2013,2623,12.9
2014,68,0.0


**Hallazgo:**

- **Vacíos en el objetivo:** 58.853 préstamos no tienen resultado final (`Current`, `Past Due`,
  `FinalPaymentInProgress`, `Cancelled`); quedan fuera.
- **Error del notebook anterior:** buscaba `Charged-Off`, que no existe. Los 11.992 `Chargedoff` se contaban como sanos.
- **Distribución:** en la población a modelar (49.552), el 31,04 % cae en impago.
- **El objetivo cambia dentro del mismo prestatario:** el 15,2 % de los 7.028 prestatarios con varios préstamos tiene
  resultados distintos entre ellos. Además, los préstamos de prestatarios repetidos caen en impago mucho menos (18,1 %)
  que los de quien tiene un solo préstamo (37,3 %).
- **Por qué importa:** si un prestatario aparece en entrenamiento y en prueba, el modelo puede apoyarse en "ya lo
  conozco". Esto confirma que hay que partir por `MemberKey`.
- **Orden temporal:** la tasa de impago cambia mucho por año, de 53,1 % en 2006 a 12,9 % en 2013. Entre los préstamos
  recientes, solo han terminado los que terminaron rápido. Es un tema abierto para la partición.

**Línea base (qué resultado obtiene una regla trivial):** "siempre pagó" → exactitud 68,96 %, recall de impago 0.
Un modelo sin información obtiene AUC-PR = 0,3104.

## 3. ¿Qué tan sucios están los datos y de qué manera?

*Preguntas:* ¿hay números guardados como texto? ¿valores centinela? ¿valores imposibles? ¿vacíos? ¿duplicados?
¿datos personales? (guía, sección 3)

In [8]:
print(df.dtypes.value_counts().to_string())
print("\nvacíos por columna (mayores, %):")
print((df.isna().mean() * 100).sort_values(ascending=False).head(12).round(2).to_string())
print("\nfilas duplicadas completas:", df.duplicated().sum())
print("ListingKey repetido (filas extra):", df[LOAN_COL].duplicated().sum())

float64    49
str        20
int64      12

vacíos por columna (mayores, %):
GroupKey                               88.29
LoanFirstDefaultedCycleNumber          85.12
ScorexChangeAtTimeOfListing            83.39
ProsperPaymentsOneMonthPlusLate        80.62
TotalProsperLoans                      80.62
ProsperPaymentsLessThanOneMonthLate    80.62
ProsperPrincipalBorrowed               80.62
TotalProsperPaymentsBilled             80.62
OnTimeProsperPayments                  80.62
ProsperPrincipalOutstanding            80.62
CreditGrade                            74.59
ClosedDate                             51.65



filas duplicadas completas: 0
ListingKey repetido (filas extra): 889


### 3.1 Texto y categorías inconsistentes: `IncomeRange`

In [9]:
print(df["IncomeRange"].map(repr).value_counts().to_string())
notebook_map = {"Not displayed", "Not Employed", "$0", "$1-24,999", "$25,000-49,999", "$50,000-74,999", "$75,000-99,999", "$100,000+"}
print("\ncategorías que el mapa de eco_score_pd.ipynb no reconoce:", sorted(set(df["IncomeRange"].dropna()) - notebook_map))

IncomeRange
'$25,000-49,999'    32192
'$50,000-74,999'    31050
'$100,000+'         17337
'$75,000-99,999'    16916
'Not displayed'      7741
'$1-24,999'          7274
'Not employed'        806
'$0 '                 621

categorías que el mapa de eco_score_pd.ipynb no reconoce: ['$0 ', 'Not employed']


### 3.2 y 3.3 Centinelas y valores imposibles

In [10]:
checks = {
    "CreditScoreRangeLower == 0": (df["CreditScoreRangeLower"] == 0).sum(),
    "StatedMonthlyIncome == 0": (df["StatedMonthlyIncome"] == 0).sum(),
    "StatedMonthlyIncome > 100.000": (df["StatedMonthlyIncome"] > 100_000).sum(),
    "DebtToIncomeRatio == 10,01 (tope)": (df["DebtToIncomeRatio"] == 10.01).sum(),
    "DebtToIncomeRatio > 1": (df["DebtToIncomeRatio"] > 1).sum(),
    "EstimatedEffectiveYield < 0": (df["EstimatedEffectiveYield"] < 0).sum(),
    "MonthlyLoanPayment == 0": (df["MonthlyLoanPayment"] == 0).sum(),
}
print(pd.Series(checks, name="filas").to_string())
cols = ["BorrowerAPR", "DebtToIncomeRatio", "StatedMonthlyIncome", "LoanOriginalAmount", "CreditScoreRangeLower"]
df[cols].describe(percentiles=[0.01, 0.5, 0.99]).T.round(3)

CreditScoreRangeLower == 0            133
StatedMonthlyIncome == 0             1394
StatedMonthlyIncome > 100.000          17
DebtToIncomeRatio == 10,01 (tope)     272
DebtToIncomeRatio > 1                 799
EstimatedEffectiveYield < 0           190
MonthlyLoanPayment == 0               935


,count,mean,std,min,1%,50%,99%,max
BorrowerAPR,113912.0,0.219,0.080,0.007,0.074,0.210,0.375,0.512
DebtToIncomeRatio,105383.0,0.276,0.552,0.000,0.030,0.220,0.860,10.010
StatedMonthlyIncome,113937.0,5608.026,7478.497,0.000,0.000,4666.667,20526.667,1750002.917
LoanOriginalAmount,113937.0,8337.014,6245.801,1000.000,1000.000,6500.000,25000.000,35000.000
CreditScoreRangeLower,113346.0,685.568,66.458,0.000,500.000,680.000,820.000,880.000


### 3.5 Vacíos que significan algo: el cambio de sistema de 2009

In [11]:
year = df["LoanOriginationQuarter"].str[-4:].astype(int)
regime = pd.DataFrame({
    "préstamos": year.value_counts().sort_index(),
    "EstimatedEffectiveYield_vacío_%": df["EstimatedEffectiveYield"].isna().groupby(year).mean().mul(100).round(1),
    "ProsperRating_vacío_%": df["ProsperRating (Alpha)"].isna().groupby(year).mean().mul(100).round(1),
    "CreditGrade_vacío_%": df["CreditGrade"].isna().groupby(year).mean().mul(100).round(1),
    "BorrowerState_vacíos": df["BorrowerState"].isna().groupby(year).sum(),
})
print(f"EstimatedEffectiveYield vacío en la población a modelar: {loans['EstimatedEffectiveYield'].isna().mean():.1%}")
regime

EstimatedEffectiveYield vacío en la población a modelar: 47.6%


,préstamos,EstimatedEffectiveYield_vacío_%,ProsperRating_vacío_%,CreditGrade_vacío_%,BorrowerState_vacíos
LoanOriginationQuarter,,,,,
2005,22,100.0,100.0,0.0,22
2006,5906,100.0,100.0,0.0,3000
2007,11460,100.0,100.0,0.0,2462
2008,11552,100.0,100.0,0.0,31
2009,2047,3.5,3.5,99.4,0
2010,5652,1.3,1.3,100.0,0
2011,11228,0.0,0.0,100.0,0
2012,19553,0.0,0.0,100.0,0
2013,34345,0.0,0.0,100.0,0


### 3.7 Fechas

In [12]:
for c in ["ListingCreationDate", "DateCreditPulled", "LoanOriginationDate", "ClosedDate"]:
    print(f"{c:22s}", df[c].dropna().head(4).tolist())
damaged = df["ListingCreationDate"].str.fullmatch(r"\d{2}:\d{2}\.\d")
print(f"\nListingCreationDate con formato 'mm:ss.d' (sin fecha): {damaged.mean():.1%}")
parsed = pd.to_datetime(df["LoanOriginationDate"], format="%d/%m/%Y %H:%M", errors="coerce")
print(f"LoanOriginationDate legible: {parsed.notna().mean():.1%} | rango: {parsed.min().date()} a {parsed.max().date()}")

ListingCreationDate    ['09:29.3', '28:07.9', '00:47.1', '02:35.0']
DateCreditPulled       ['41:46.8', '27/02/2014 8:28', '09:10.1', '22/10/2012 11:02']
LoanOriginationDate    ['12/09/2007 0:00', '3/03/2014 0:00', '17/01/2007 0:00', '1/11/2012 0:00']
ClosedDate             ['14/08/2009 0:00', '17/12/2009 0:00', '7/01/2008 0:00', '19/12/2012 0:00']



ListingCreationDate con formato 'mm:ss.d' (sin fecha): 99.7%


LoanOriginationDate legible: 100.0% | rango: 2005-11-15 a 2014-03-12


### 3.8 Categorías raras y el índice climático

In [13]:
CLIMATE_INDEX = {"FL": 94.2, "LA": 91.0, "TX": 83.5, "NC": 77.1, "GA": 70.4,
                 "NY": 56.8, "CA": 48.0, "OH": 40.2, "NV": 14.1, "UT": 10.5}  # matriz de eco_score_pd.ipynb
states = df["BorrowerState"].value_counts()
print(f"estados distintos: {states.size} | con menos de 100 filas: {states[states < 100].to_dict()}")
print(f"Occupation: {df['Occupation'].nunique()} valores")
with_state = df["BorrowerState"].notna()
default_50 = with_state & ~df["BorrowerState"].isin(CLIMATE_INDEX)
print(f"estados sin índice (reciben 50,0): {states.size - len(CLIMATE_INDEX)} "
      f"| filas: {default_50.sum():,} de {with_state.sum():,}")

estados distintos: 51 | con menos de 100 filas: {'ND': 52}
Occupation: 67 valores
estados sin índice (reciben 50,0): 41 | filas: 58,204 de 108,422


**Hallazgos de calidad (con cifras):**

- **Repetidos:** `df.duplicated()` da 0, pero `ListingKey` se repite en 889 filas. La comparación debe hacerse por
  llave, no por fila completa.
- **`IncomeRange`:** `'$0 '` trae un espacio al final (621 filas) y `'Not employed'` (806) no coincide con el mapa del
  notebook anterior (`'Not Employed'`). `Not displayed` (7.741) no significa "sin ingresos".
- **Valores imposibles o en el tope:**
  - `CreditScoreRangeLower` = 0 en 133 filas.
  - `StatedMonthlyIncome` > 100.000 USD/mes en 17 filas.
  - `DebtToIncomeRatio` = 10,01 en 272 filas: es el tope del sistema.
  - `MonthlyLoanPayment` = 0 en 935 filas: hay un préstamo pero la cuota es 0. Queda abierto.
- **Cambio de sistema en 2009:**
  - `EstimatedEffectiveYield` y `ProsperRating` están 100 % vacíos de 2005 a 2008; `CreditGrade`, al revés.
  - En la población a modelar, `EstimatedEffectiveYield` está vacío en el 47,6 % de las filas.
- **`BorrowerState`:** vacío en 5.515 filas, casi todas de 2006–2007.
- **Fechas:**
  - `ListingCreationDate` llega sin fecha (`mm:ss.d`) en el 99,7 % de las filas.
  - `DateCreditPulled` mezcla ese formato con fechas completas.
  - `LoanOriginationDate` se lee al 100 % (15/11/2005 a 12/03/2014).
- **Índice climático:** 41 de 51 estados reciben el valor por defecto 50,0.

## 4. ¿Qué variables deberían ser estables y no lo son?

Dos niveles:

- **Dentro de un mismo préstamo (`ListingKey` repetido):** todo debería ser idéntico.
- **Dentro de un mismo prestatario:** sus préstamos son de fechas distintas, así que un cambio puede ser real (se mudó,
  cambió de empleo).

In [14]:
dup = df[df[LOAN_COL].duplicated(keep=False)]
changes_in_loan = (dup.groupby(LOAN_COL).nunique(dropna=False) > 1).mean().sort_values(ascending=False)
print(f"ListingKey repetidos: {dup[LOAN_COL].nunique()} | % en que cada columna cambia (top 5):")
print((changes_in_loan.head(5) * 100).round(1).to_string())

ListingKey repetidos: 844 | % en que cada columna cambia (top 5):
ProsperScore           99.3
ListingNumber           2.1
BorrowerAPR             2.1
ListingCreationDate     2.1
LenderYield             2.1


In [15]:
stable_candidates = ["BorrowerState", "Occupation", "IsBorrowerHomeowner", "IncomeRange", "EmploymentStatus"]
multi_rows = df[df[ID_COL].isin(multi.index)].drop_duplicates(subset=LOAN_COL)
changes = {c: multi_rows.groupby(ID_COL)[c].nunique().gt(1).mean() for c in stable_candidates}
pd.Series(changes, name="% de prestatarios (con >1 préstamo) con más de un valor").mul(100).round(1)

BorrowerState           4.7
Occupation             11.2
IsBorrowerHomeowner     8.1
IncomeRange            30.2
EmploymentStatus       30.6
Name: % de prestatarios (con >1 préstamo) con más de un valor, dtype: float64

**Hallazgo:**

- **Dentro de un mismo préstamo:** en 844 `ListingKey` repetidos, el 99,3 % solo cambia `ProsperScore`. Las demás
  columnas cambian en un 2,1 % de los casos.
- **Dentro de un mismo prestatario con más de un préstamo**, porcentaje que tiene más de un valor:

  | Variable | % con más de un valor |
  |---|---|
  | `BorrowerState` | 4,7 % |
  | `IsBorrowerHomeowner` | 8,1 % |
  | `Occupation` | 11,2 % |
  | `IncomeRange` | 30,2 % |
  | `EmploymentStatus` | 30,6 % |

**¿Ruido o cambio real? ¿Por qué?**

- **Repetidos de un mismo préstamo:** es **ruido**, porque es el mismo préstamo. Se deja una fila (decisión 2).
- **Entre préstamos de un mismo prestatario:** es plausible que sea **cambio real**. Son préstamos pedidos en fechas
  distintas, y en años la persona puede mudarse, cambiar de empleo o ganar más. Por eso **no se repara**: cada préstamo
  conserva los datos que tenía al otorgarse.
  - El índice climático también se asigna con el estado de **ese** préstamo.
  - El 30 % de cambio en `EmploymentStatus` es alto. Queda por revisar si Prosper cambió las categorías con el tiempo,
    antes de usarla como variable.

## 5. ¿Hay algo que suene a fuga de información?

La decisión que apoyamos ocurre **al otorgar el préstamo**. Toda columna que se llena después contiene, total o
parcialmente, la respuesta. Lo comprobamos comparando estas columnas entre préstamos pagados y en impago.

In [16]:
post_origination = ["LP_CustomerPayments", "LP_CustomerPrincipalPayments", "LP_InterestandFees", "LP_ServiceFees",
                    "LP_CollectionFees", "LP_GrossPrincipalLoss", "LP_NetPrincipalLoss", "LP_NonPrincipalRecoverypayments",
                    "ClosedDate", "LoanCurrentDaysDelinquent", "LoanFirstDefaultedCycleNumber", "LoanMonthsSinceOrigination"]
print(f"columnas posteriores al otorgamiento: {len(post_origination)}")

evidence = pd.DataFrame({
    "pagó": [(loans.loc[loans[TARGET] == 0, "LP_NetPrincipalLoss"] > 0).mean(),
             loans.loc[loans[TARGET] == 0, "LoanFirstDefaultedCycleNumber"].notna().mean(),
             (loans.loc[loans[TARGET] == 0, "LoanCurrentDaysDelinquent"] > 0).mean()],
    "impago": [(loans.loc[loans[TARGET] == 1, "LP_NetPrincipalLoss"] > 0).mean(),
               loans.loc[loans[TARGET] == 1, "LoanFirstDefaultedCycleNumber"].notna().mean(),
               (loans.loc[loans[TARGET] == 1, "LoanCurrentDaysDelinquent"] > 0).mean()],
}, index=["LP_NetPrincipalLoss > 0", "LoanFirstDefaultedCycleNumber con valor", "LoanCurrentDaysDelinquent > 0"])
(evidence * 100).round(1)

columnas posteriores al otorgamiento: 12


,pagó,impago
LP_NetPrincipalLoss > 0,0.0,97.8
LoanFirstDefaultedCycleNumber con valor,0.1,99.3
LoanCurrentDaysDelinquent > 0,0.0,100.0


In [17]:
# Puntajes propios de Prosper: no son posteriores, pero pueden resumir casi todo el riesgo. Se miden con y sin ellos.
prosper_scores = ["ProsperRating (numeric)", "ProsperScore", "EstimatedLoss", "EstimatedReturn"]
post2009 = loans[loans["ProsperRating (numeric)"].notna()]
print(f"préstamos con puntaje de Prosper (2009 en adelante): {len(post2009):,}")
post2009.groupby("ProsperRating (Alpha)")[TARGET].agg(préstamos="size", impago_pct="mean").assign(
    impago_pct=lambda t: (t["impago_pct"] * 100).round(1)).sort_values("impago_pct")

préstamos con puntaje de Prosper (2009 en adelante): 25,990


,préstamos,impago_pct
ProsperRating (Alpha),,
AA,1781,4.7
A,3606,11.2
B,3371,17.4
C,3812,22.0
D,5868,28.6
E,3828,34.6
HR,3724,38.2


**Columnas sospechosas de fuga y por qué:**

- **Fuga segura (se excluyen al cargar):** las 8 `LP_*`, `ClosedDate`, `LoanCurrentDaysDelinquent`,
  `LoanFirstDefaultedCycleNumber` y `LoanMonthsSinceOrigination`. Se conocen después de otorgar el préstamo, y la tabla
  de arriba muestra que separan casi perfectamente pagó de impago. `LoanStatus` es la fuente del objetivo.
- **Del notebook anterior:**
  - `Target_Risk_Scale` y `Financial_Status` se derivan del objetivo y del índice climático.
  - `BorrowerState_Encoded` se calculaba con todo el dataset; debe ir dentro del `Pipeline`.
- **A medir con y sin ellas en la etapa 4:**
  - `ProsperRating`, `ProsperScore`, `EstimatedLoss` y `EstimatedReturn` son el puntaje de riesgo de Prosper. La tasa
    de impago crece con el rating (tabla de arriba), así que pueden dominar el modelo.
  - `Investors` y `PercentFunded` se conocen al cerrar la financiación; hay que confirmar que están disponibles al decidir.

## 6. Tabla de decisiones (entregable del EDA)

Copiada en `reports/01_eda.md` y en `especificaciones/02-eda.md` (con la función que implementará cada una).

| # | Hallazgo (con cifra) | Decisión | Motivo |
|---|---|---|---|
| 1 | `Target_Risk_Scale` depende del índice climático en el 94,0 % de las filas y 10.952 `Chargedoff` quedaron como "Saludable" | Objetivo `is_default` (aprobado el 30 oct) (1 = `Chargedoff`/`Defaulted`, 0 = `Completed`); se excluyen los 58.853 préstamos sin resultado final | El resultado de un préstamo sin terminar no se conoce, y el objetivo no puede construirse con una variable del modelo |
| 2 | 844 `ListingKey` repetidos (889 filas extra); el 99,3 % solo difiere en `ProsperScore` | Dejar la primera fila por `ListingKey` | Es el mismo préstamo con doble peso |
| 3 | 15.744 prestatarios con varios préstamos (7.028 en la población a modelar); el 15,2 % de estos últimos tiene resultados distintos entre sus préstamos | Partir por `MemberKey` | Evitar que un prestatario quede en entrenamiento y en prueba |
| 4 | `BorrowerState` vacío en 5.515 filas (4,84 %), casi todas de 2006–2007 | Eliminar esas filas | Sin estado no hay índice climático |
| 5 | 12 columnas posteriores al otorgamiento separan pagó de impago | Excluirlas al cargar | Contienen la respuesta |
| 6 | `IncomeRange`: `'$0 '` (621), `'Not employed'` (806), `Not displayed` (7.741) | Quitar espacios y mapear explícitamente; `Not displayed` como categoría propia | "No mostrado" ≠ "sin ingresos" |
| 7 | `CreditScoreRangeLower` = 0 (133); `StatedMonthlyIncome` > 100.000 (17) | Convertir en vacío | Valores imposibles |
| 8 | `EstimatedEffectiveYield` y `ProsperRating` vacíos en el 100 % de 2005–2008 (47,6 % de la población) | Dejar el vacío; no imputar fuera del `Pipeline` | El vacío significa "la métrica no existía" |
| 9 | `ListingCreationDate` y `DateCreditPulled` sin fecha (`mm:ss.d`) | No usarlas; fechar con `LoanOriginationDate` | No son recuperables |
| 10 | 41 de 51 estados con índice por defecto 50,0 | Índice climático solo como variable; medir con y sin él | Es una hipótesis a probar |

## Entregable de esta etapa

- [x] Este notebook ejecuta de principio a fin.
- [x] La tabla de decisiones tiene **al menos 5** filas con cifra y motivo (10).
- [x] `reports/01_eda.md` está actualizado.
- [x] Declaré mi **unidad de análisis** por escrito (un préstamo; partición por `MemberKey`).
- [x] El grupo aprobó el objetivo `is_default` (decisión 1), el 30 oct.

**Siguiente:** `03_preprocesamiento.ipynb`.